# vv4ml Quickstart Guide
Welcome to the introduction tutorial of vv4ml library.

This tutorial will show you how to:
1. install vv4ml
2. generate a mock dataset and train a model of reference
3. run **convergence**, **numerical error** and **smoothness** tests to assess your model credibility.

## Installation

In [ ]:
!pip install vv4ml

## Environment setup and data generating
To run vv4ml, let's generate some datas with `scikit-learn` and train a model of reference (in this example `RandomForestRegressor`).

In [6]:
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor

# 1. Mock dataset generation (3 feature and 1 target)
x_raw, y_raw = make_regression(n_samples=500, n_features=3, noise=0.1, random_state=42)

# Shift data away from zero to stabilize the error computation
y_raw = y_raw + 1000

# Pandas structure conversion (necessary to be compatible with vv4ml)
x_train = pd.DataFrame(x_raw[:400], columns=['feature_1', 'feature_2', 'feature_3'])
y_train_series = pd.Series(y_raw[:400])
y_train_df = pd.DataFrame(y_raw[:400], columns=['target']) # This is necessary to the numerical error test

x_eval = pd.DataFrame(x_raw[400:], columns=['feature_1', 'feature_2', 'feature_3'])
y_eval = y_raw[:400]

# 2. Model of reference training
model_type = "RandomForestRegressor"
best_params = {"n_estimators": 50, "max_depth": 5, "random_state": 42}

ref_model = RandomForestRegressor(**best_params)
ref_model.fit(x_train, y_train_series)

# Reference predictions extraction
ref_predictions = ref_model.predict(x_eval)

print(f"Train dataset: {x_train.shape}")
print(f"Evaluation dataset: {x_eval.shape}")

Train dataset: (400, 3)
Evaluation dataset: (100, 3)


## Convergence test

In [7]:
from vv4ml.verification.convergence import run_convergence_test

# Test configuration
n_rows_to_test = [50, 100, 200, 300]
threshold_convergence = 0.15 # This is the 15% of the maximum relative error tolerance related to the complete model

# Run
convergence_result = run_convergence_test(
    x_train=x_train,
    y_train=y_train_series,
    x_eval=x_eval,
    ref_predictions=ref_predictions,
    model_type=model_type,
    best_params=best_params,
    n_rows=n_rows_to_test,
    threshold=threshold_convergence,
    metric="MaximumRelativeError"
)

# Output
print(f"-- Convergence test result ---")
print(f"Test outcome: {convergence_result.passed}")
print(f"Tested subsets (k): {convergence_result.n_rows_tested}")
print(f"Captured errors: {convergence_result.errors}")
print(f"Threshold: {convergence_result.threshold}")
print(f"Minimum k value for convergence: {convergence_result.min_k}")

-- Convergence test result ---
Test outcome: True
Tested subsets (k): [50, 100, 200, 300]
Captured errors: [0.11512080634644978, 0.09869312582982319, 0.045136549359861645, 0.03826715098144992]
Threshold: 0.15
Minimum k value for convergence: 50


## Numerical Error Test

In [10]:
from vv4ml.verification.numerical_error import run_numerical_error_test

# Use the minimum k found in the previous convergence test as the row size. Otherwise, fallback to 100
convergence_k = convergence_result.min_k if convergence_result.min_k else 100

numerical_error_result = run_numerical_error_test(
    x_train=x_train,
    y_train=y_train_df,
    x_eval=x_eval,
    model_type=model_type,
    best_params=best_params,
    n_repetitions=5,
    convergence_rows=convergence_k,
    threshold=1.0,
    random_state=42
)

# Output
print(f"-- Numerical error result ---")
print(f"Test outcome: {numerical_error_result.passed}")
print(f"Maximum standard deviation found: {numerical_error_result.max_variance}")
print(f"Maximum threshold: {numerical_error_result.threshold}")
print(f"Number of ran repetitions: {numerical_error_result.repetitions}")

-- Numerical error result ---
Test outcome: False
Maximum standard deviation found: 87.95002294382134
Maximum threshold: 1.0
Number of ran repetitions: 5


/home/cittian/Documents/vv4ml/.venv/lib64/python3.14/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/cittian/Documents/vv4ml/.venv/lib64/python3.14/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/cittian/Documents/vv4ml/.venv/lib64/python3.14/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/cittian/Documents/vv4ml/.venv/lib64/python3.14/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vec

## Smoothness test

In [11]:
from vv4ml.verification.smoothness import run_smoothness_test

# Compute real dataset evaluation ranges
input_ranges = [(float(x_eval[col].min()), float(x_eval[col].max())) for col in x_eval.columns]

smoothness_result = run_smoothness_test(
    model=ref_model,
    x_eval=x_eval,
    input_ranges=input_ranges,
    sample_fraction=0.2,                # Test 20% of evaluation points
    perturbation_scaled_magnitude=1e-4, # Delta perturbation
    threshold=2.5,                      # Maximum threshold of the relative error ratio
    random_state=42
)

# Output

print(f"-- Smoothness result ---")
print(f"Test outcome: {smoothness_result.passed}")
print(f"Maximum error ratio: {smoothness_result.max_error_ratio}")
print(f"Threshold: {smoothness_result.threshold}")

-- Smoothness result ---
Test outcome: True
Maximum error ratio: 0.08857963750799089
Threshold: 2.5
